In [0]:
import yfinance as yf
import pandas as pd

stocks = spark.sql(f"""select distinct company_stock_symbol from operations.finance.dim_company where sp_500_indicator = 1""")

stock_list = [row['company_stock_symbol'] for row in stocks.select('company_stock_symbol').collect()]

stock_data_untransposed = yf.download(stock_list, start='2020-01-01', end='2026-08-25', auto_adjust=False)

stock_data = (
    stock_data_untransposed
    .stack(level=1, future_stack=True)
    .reset_index()
    .rename(columns={'level_1': 'stock_symbol'})
)

df_date = spark.sql(f"""select distinct * from operations.finance.dim_date""").toPandas()
df_company = spark.sql(f"""select distinct company_bigint_key,company_stock_symbol from operations.finance.dim_company""").toPandas()

df_date['date_value'] = pd.to_datetime(df_date['date_value'])

final_all_columns = stock_data.merge(df_date, left_on='Date', right_on='date_value').merge(df_company, left_on='Ticker', right_on='company_stock_symbol')
final_not_renamed = final_all_columns[['date_key', 'company_bigint_key', 'Adj Close', 'Close','High','Low','Open','Volume']]
final = final_not_renamed.rename(columns = {'Adj Close':'adj_close', 'Close':'close', 'High':'high', 'Low':'low', 'Open':'open', 'Volume':'volume'})

df_stocks = spark.createDataFrame(final)

In [0]:
indices = ['^GSPC', '^IXIC', '^DJI']

index_data_untransposed = yf.download(indices, start='2020-01-01', end='2026-08-25', auto_adjust=False)

index_data = (
    index_data_untransposed
    .stack(level=1, future_stack=True)
    .reset_index()
    .rename(columns={'level_1': 'stock_symbol'})
)

df_date = spark.sql(f"""select distinct * from operations.finance.dim_date""").toPandas()
# df_company = spark.sql(f"""select distinct company_bigint_key,company_stock_symbol from operations.finance.dim_company""").toPandas()

df_date['date_value'] = pd.to_datetime(df_date['date_value'])

company_key_map = {'^GSPC': 1, '^IXIC': 2, '^DJI': 3}
index_data['company_bigint_key'] = index_data['Ticker'].map(company_key_map).astype('int64')

final_all_columns = index_data.merge(df_date, left_on='Date', right_on='date_value')

final_not_renamed = final_all_columns[['date_key', 'company_bigint_key', 'Adj Close', 'Close','High','Low','Open','Volume']]
final = final_not_renamed.rename(columns = {'Adj Close':'adj_close', 'Close':'close', 'High':'high', 'Low':'low', 'Open':'open', 'Volume':'volume'})

df_indices = spark.createDataFrame(final)

In [0]:
df = df_stocks.union(df_indices)
df.write.mode('overwrite').saveAsTable('operations.finance.fact_price_daily')